# Digit addition: a from-scratch transformer, and its addition circuit

## What this project is

Train a small transformer from scratch on **decimal addition**, then reverse-engineer the
algorithm it learns. This extends Nanda et al.'s *Progress measures for grokking via
mechanistic interpretability*, which analysed **modular** arithmetic and found the model had
learned a Fourier/trigonometric algorithm. Ordinary multi-digit addition is a different problem
-- it is positional and needs carrying -- so whatever circuit appears should be structurally
different, and that difference is the contribution.

## Why it is set up this way

Notebook 1 (`1_transformer_from_scratch.ipynb`) was the precursor: build a GPT-2-shaped
transformer by hand and prove it trains. That is done. Everything below the "carried over"
heading is that code, unchanged apart from dropping ARENA's test harness.

The important change here is the **vocabulary**: 12 symbols (`0`-`9`, `+`, `=`) instead of
50257. In notebook 1, 97% of the parameters were the embedding table and only 3% did any
computation. Here it inverts -- essentially every parameter computes. That is what makes the
interpretability tractable: there is an actual algorithm inside, not a lookup table.

## The minimum result

**Identify which attention heads implement the carry, via ablation plus attention patterns.**

Anything beyond that -- the full mechanism, a comparison of circuits across digit positions,
grokking dynamics -- is bonus. This phase has no natural end, so the floor matters.


In [ ]:
from dataclasses import dataclass

import einops
import torch as t
from jaxtyping import Int
from torch import Tensor
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print("device:", device)

---

# What to build next

## 1. The task encoding

Decide and write down:

- **Vocabulary** -- `0`-`9`, `+`, `=` is 12 symbols. A dict, not a trained tokenizer.
- **Sequence format** -- e.g. `49+97=641`, fixed length so every example is the same shape.
  Note the answer is **reversed**: the model emits the least significant digit first, so carries
  propagate in the same direction as generation. Writing it forwards asks the model to predict
  the leading digit before it has computed the carries that determine it.
- **Zero-padding** -- `05` not `5`, so digit positions mean the same thing in every example.
- **Number of digits** -- 2 is enough to have carries and small enough to enumerate exhaustively
  (100x100 = 10,000 pairs).

In [ ]:
VOCABULARY = {f"{i:d}": i for i in range(10)}
VOCABULARY["+"] = 10
VOCABULARY["="] = 11

# convenience aliases
TOKEN_TO_ID = VOCABULARY
ID_TO_TOKEN = {idx: token for token, idx in TOKEN_TO_ID.items()}
D_VOCAB = len(VOCABULARY)

# 1D (seq,) or 2D (batch, seq) ids, as a tensor or plain lists
TokenIds = Tensor | list[int] | list[list[int]]


def to_tokens(strings: str | list[str]) -> Int[Tensor, "batch seq"]:
    """'49+97=641' -> tensor([[4, 9, 10, 9, 7, 11, 6, 4, 1]]). Always returns (batch, seq)."""
    if isinstance(strings, str):
        strings = [strings]
    return t.tensor([[TOKEN_TO_ID[c] for c in s] for s in strings], device=device)


def to_str_tokens(tokens: TokenIds) -> list[str] | list[list[str]]:
    """Token ids -> the characters they stand for, one per token. Nested if the input is 2D."""
    if isinstance(tokens, Tensor):
        tokens = tokens.tolist()
    if tokens and isinstance(tokens[0], list):
        return [[ID_TO_TOKEN[i] for i in row] for row in tokens]
    return [ID_TO_TOKEN[i] for i in tokens]


def to_string(tokens: TokenIds) -> str | list[str]:
    """Token ids -> '49+97=641'. A list of strings if the input is 2D."""
    strs = to_str_tokens(tokens)
    if strs and isinstance(strs[0], list):
        return ["".join(row) for row in strs]
    return "".join(strs)


# sequences will have the result of the computation reversed


## 2. Dataset

Enumerate every `(a, b)` pair and split. No `datasets`, no Arrow, no DataLoader workers --
10,000 examples of 9 tokens fits in memory as a single tensor.

**The split fraction is the most consequential knob.** Hold out a large fraction and you may
reproduce **grokking**: the model memorises the training pairs, sits at chance on the held-out
ones, then abruptly generalises thousands of steps later. That is Nanda's phenomenon, and
producing it on this task is a result in itself. Worth running a sweep over the fraction.

Also decide: **compute loss only on the answer positions**. The question digits are given, so
predicting them is free and would dilute the metric.

In [ ]:
def sum_dataset(num_digits: int = 2, seed: int = 42, frac_train: float = 0.3) -> tuple[Tensor, Tensor]:
    
    pairs = []

    for i in range(10**num_digits):
        for j in range(10**num_digits):
            pairs.append(f"{i:0{num_digits}d}+{j:0{num_digits}d}={f'{(i+j):0{num_digits+1}d}'[::-1]}")

    data = t.tensor([[TOKEN_TO_ID[token] for token in example] for example in pairs], device=device)
    g = t.Generator().manual_seed(seed)

    _n = data.shape[0]
    split = t.randperm(_n, generator=g)
    train_idx = split[:int(frac_train * _n)].to(device=device)
    test_idx = split[int(frac_train * _n):].to(device=device)


    assert t.cat([train_idx, test_idx], dim=0).sort().values.equal(t.arange(len(data), device=data.device))

    train = data[train_idx]
    test = data[test_idx]
    
    return train, test
train, test = sum_dataset()
print(f"train: {train.shape} \ntest: {test.shape}")

## 3. Train

Use a `HookedTransformer` from TransformerLens rather than the `DemoTransformer` above.

You have already proven you can write the internals -- that was notebook 1. Building the
project model as a `HookedTransformer` gets you hook points for free and skips debugging a
`state_dict` mapping later. Keep `DemoTransformer` here as reference.

Sizing: this model should be *tiny*. 1-2 layers, small `d_model`, `d_vocab=12`. Training runs
in minutes, so iterate freely.

### Training

I'll start by copying their TrainArgs, then write my own training infra and modify TrainArgs as necessary to fit my objectives.

In [ ]:
@dataclass
class TrainArgs:
    num_digits: int
    # trainset_size: int   -- derivable from train.shape[0]
    # valset_size: int     -- derivable from test.shape[0]
    frac_train: float
    epochs: int
    # batch_size: int      -- full batch, so there is nothing to vary
    lr: float
    # lr_start: float
    # lr_end: float
    weight_decay: float
    seed: int
    eval_every: int
    # d_model: int
    # d_head: int
    # n_layers: int
    # n_heads: int
    # d_mlp: int
    # normalization_type: str | None
    use_wandb: bool
    device: str

    def __post_init__(self):
        self.seq_len = self.num_digits * 3 + 2 + 1  # We have [{a}, +, {b}, =, {a+b}]

In [ ]:
args = TrainArgs(
    num_digits=2,
    frac_train=0.3,
    epochs=40000,
    lr=1e-3,
    weight_decay=1.0,
    seed=42,
    eval_every=10,
    use_wandb=False,
    device=device,
)

### Defining the model

In [ ]:
def create_model(args: TrainArgs) -> HookedTransformer:
    """A fresh model from init, sized from `args`.

    Re-seeded on every call, so a sweep over `frac_train` varies only the thing
    being swept -- every run starts from identical weights.
    """
    t.manual_seed(args.seed)

    d_model = 128
    n_heads = 4

    cfg = HookedTransformerConfig(
        n_layers=2,
        n_ctx=args.seq_len,  # We have [{a}, +, {b}, =, {a+b}]
        d_model=d_model,
        d_head=d_model // n_heads,
        n_heads=n_heads,
        d_mlp=4 * d_model,
        attn_only=False,
        act_fn="relu",
        # We have all digits, and "+="
        d_vocab=len(VOCABULARY),
        # it's a small transformer so may as well use these hooks
        use_attn_result=True,
        use_split_qkv_input=True,
        use_hook_tokens=True,
        # Layernorm makes things way more accurate, even though it makes
        # mech interp a little more annoying!
        normalization_type=None,
        device=args.device,
    )

    return HookedTransformer(cfg)


model = create_model(args)
cfg = model.cfg

In [ ]:
def param_breakdown(model: HookedTransformer, n_train: int | None = None) -> None:
    """Parameter counts grouped by component, with the attention:MLP ratio.

    The ratio is worth watching more than the total -- attention routes information
    between positions, MLPs do the arithmetic, and a config that starves the MLPs
    will struggle on this task. The standard d_mlp = 4 * d_model gives ~0.50x.
    """
    groups: dict[str, int] = {}
    for name, param in model.named_parameters():
        key = "attn" if ".attn." in name else "mlp" if ".mlp." in name else "embed/unembed"
        groups[key] = groups.get(key, 0) + param.numel()

    total = sum(groups.values())
    for key, count in sorted(groups.items(), key=lambda kv: -kv[1]):
        print(f"  {key:16s} {count:>10,}  ({count / total * 100:5.1f}%)")
    print(f"  {'total':16s} {total:>10,}")

    if groups.get("mlp"):
        print(f"\n  attn:mlp ratio   {groups['attn'] / groups['mlp']:.2f}x")
    if n_train:
        print(f"  params per training example   {total / n_train:.0f}")


param_breakdown(model)


In [ ]:
for name, param in model.named_parameters():
    print(f"{name:<20} {str(param.shape):<20} {param.numel()}")
sum(p.numel() for p in model.parameters())      # 584,556

In [ ]:
print(
    train.shape,
    to_string(train[t.randint(low=0, high=len(train), size=(1,))]),
    sep="\n",
)

In [ ]:
class Trainer:
    def __init__(self, model: HookedTransformer, args: TrainArgs):
        self.args = args
        self.model = model
        
    def training_step(self, toks: Tensor) -> Tensor:
        logits, target = self._shared_train_validation_step(toks)
        return self._loss(logits, target)
    
    def validation_step(self, toks: Tensor) -> tuple[float, float]:
        logits, target = self._shared_train_validation_step(toks)
        loss = self._loss(logits, target).item()
        accuracy = (logits.argmax(dim=-1) == target).float().mean().item()
        return loss, accuracy
    
    def _loss(self, logits: Tensor, target: Tensor) -> Tensor:
        return t.nn.functional.cross_entropy(
            einops.rearrange(logits, "batch seq vocab_out -> (batch seq) vocab_out"),
            einops.rearrange(target, "batch seq -> (batch seq)")
        )
    
    def _shared_train_validation_step(self, toks: Tensor) -> tuple[Tensor, Tensor]:
        toks = toks.to(self.args.device)
        logits = self.model(toks)[:, -(self.args.num_digits + 2): -1]
        target = toks[:, -(self.args.num_digits + 1):]
        return logits, target
    
    def configure_optimizers(self):
        optimizer = t.optim.AdamW(
            self.model.parameters(), lr=self.args.lr, weight_decay=self.args.weight_decay
        )
        return optimizer

def train_model(model: HookedTransformer, args: TrainArgs):
    trainer = Trainer(model=model, args=args)
    optimizer = trainer.configure_optimizers()
    
    train, test = sum_dataset(
        num_digits=args.num_digits, seed=args.seed, frac_train=args.frac_train
    )

    # the two curves that make the grokking plot, plus test accuracy
    history = {"step": [], "train_loss": [], "test_loss": [], "test_accuracy": []}

    # training
    optimizer.zero_grad()
    progress_bar = tqdm(range(args.epochs))
    for epoch in progress_bar:
        loss = trainer.training_step(train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
        # validation
        if epoch % args.eval_every == 0 or epoch == args.epochs - 1:
            with t.inference_mode():
                test_loss, test_accuracy = trainer.validation_step(test)
            history["step"].append(epoch)
            history["train_loss"].append(loss.item())
            history["test_loss"].append(test_loss)
            history["test_accuracy"].append(test_accuracy)
            progress_bar.set_description(
                f"train {loss.item():.4f} | test {test_loss:.4f} | acc {test_accuracy:.3f}"
            )

    return trainer.model, history


## 4. Interpretability

Roughly in order of cost:

- **Attention patterns** on carry vs non-carry examples (`circuitsvis`). Do specific heads look
  at the digit positions that generate a carry?
- **Per-head ablation** -- zero `hook_z` for each head in turn, measure the accuracy drop. This
  tells you *which* heads matter, not *how* they work.
- **Direct logit attribution** -- which component actually writes the answer digit.
- **The mechanism** -- how does carry information move between digit positions?

Reference: `chapter1_transformer_interp/exercises/part52_grokking_and_modular_arithmetic` is
ARENA's own exercise on the Nanda result, already downloaded. Read it before this phase.

Hook names and shapes: `notes/img/1_architecture/transformerlens-hook-names-annotated_cell059.png`.

**Watch out for attention sinks.** These sequences have no BOS token, so heads with nothing
useful to attend to will dump their attention on whatever is at position 0 or on the `=`.
A head that appears to "attend to the equals sign" for no clear reason is probably a sink,
not a circuit.

## 5. Write-up

`README.md` and `results/`, both already scaffolded in the repo. Motivation, method, findings,
plots. Compare the circuit you find against Nanda's modular-arithmetic one -- that contrast is
the point of the whole project.